# 01. Persiapan Data

Tujuan notebook ini:
1. Membaca 11 file BPS (2016–2026), mengambil baris Jabodetabek
2. Memvalidasi hasil parsing (jumlah 12 bulan vs kolom Tahunan)
3. Membersihkan data kalender
4. Menggabungkan target + kalender, lalu menyimpan ke `data/processed/`

In [1]:
import os
import sys
import pandas as pd

# supaya bisa import file di folder src
sys.path.append(os.path.abspath("../src"))
import data_prep as dp

# lokasi folder
FOLDER_RAW = "../data/raw"
FOLDER_PROCESSED = "../data/processed"
os.makedirs(FOLDER_PROCESSED, exist_ok=True)

print("Folder raw      :", os.path.abspath(FOLDER_RAW))
print("Folder processed:", os.path.abspath(FOLDER_PROCESSED))

Folder raw      : c:\KULIAH\SEM 5\Project Sistem Cerdas\Prediksi KRL\prediksi-penumpang-krl-jabodetabek\data\raw
Folder processed: c:\KULIAH\SEM 5\Project Sistem Cerdas\Prediksi KRL\prediksi-penumpang-krl-jabodetabek\data\processed


In [2]:
for nama in sorted(os.listdir(FOLDER_RAW)):
    ukuran = os.path.getsize(os.path.join(FOLDER_RAW, nama))
    print(f"{nama:55s} {ukuran:>8d} byte")

.gitkeep                                                       6 byte
Jumlah Penumpang Kereta Api, 2016.csv                        784 byte
Jumlah Penumpang Kereta Api, 2017.csv                        784 byte
Jumlah Penumpang Kereta Api, 2018.csv                        784 byte
Jumlah Penumpang Kereta Api, 2019.csv                        784 byte
Jumlah Penumpang Kereta Api, 2020.csv                        787 byte
Jumlah Penumpang Kereta Api, 2021.csv                        791 byte
Jumlah Penumpang Kereta Api, 2022.csv                        806 byte
Jumlah Penumpang Kereta Api, 2023.csv                        806 byte
Jumlah Penumpang Kereta Api, 2024.csv                        940 byte
Jumlah Penumpang Kereta Api, 2025.csv                        940 byte
Jumlah Penumpang Kereta Api, 2026.csv                        796 byte
kalender_eksogen_krl_2013_2026.csv                          5843 byte


In [3]:
# uji dulu pada satu file sebelum membaca semuanya
path_uji = os.path.join(FOLDER_RAW, "Jumlah Penumpang Kereta Api, 2020.csv")
hasil_uji = dp.baca_satu_file(path_uji)

print("Tahun        :", hasil_uji["tahun"])
print("Tahunan resmi:", hasil_uji["tahunan_resmi"])
print(hasil_uji["tabel"])
print("Jumlah 12 bulan:", hasil_uji["tabel"]["penumpang_ribu"].sum())

Tahun        : 2020
Tahunan resmi: 154591.0
   year_month  penumpang_ribu
0     2020-01         26733.0
1     2020-02         25616.0
2     2020-03         18548.0
3     2020-04          5138.0
4     2020-05          5077.0
5     2020-06          8591.0
6     2020-07         11116.0
7     2020-08         11014.0
8     2020-09          9678.0
9     2020-10         10128.0
10    2020-11         11622.0
11    2020-12         11330.0
Jumlah 12 bulan: 154591.0


In [4]:
target_mentah, tabel_validasi = dp.gabung_semua(FOLDER_RAW)

print()
print("Ukuran target gabungan:", target_mentah.shape)
print(target_mentah.head(15))
print("...")
print(target_mentah.tail(15))

Ditemukan 11 file.
Membaca: Jumlah Penumpang Kereta Api, 2016.csv
Membaca: Jumlah Penumpang Kereta Api, 2017.csv
Membaca: Jumlah Penumpang Kereta Api, 2018.csv
Membaca: Jumlah Penumpang Kereta Api, 2019.csv
Membaca: Jumlah Penumpang Kereta Api, 2020.csv
Membaca: Jumlah Penumpang Kereta Api, 2021.csv
Membaca: Jumlah Penumpang Kereta Api, 2022.csv
Membaca: Jumlah Penumpang Kereta Api, 2023.csv
Membaca: Jumlah Penumpang Kereta Api, 2024.csv
Membaca: Jumlah Penumpang Kereta Api, 2025.csv
Membaca: Jumlah Penumpang Kereta Api, 2026.csv

Ukuran target gabungan: (132, 2)
   year_month  penumpang_ribu
0     2016-01         22238.0
1     2016-02         21229.0
2     2016-03         23206.0
3     2016-04         23149.0
4     2016-05         24401.0
5     2016-06         23821.0
6     2016-07         21574.0
7     2016-08         23923.0
8     2016-09         23570.0
9     2016-10         24533.0
10    2016-11         24104.0
11    2016-12         24841.0
12    2017-01         24185.0
13    2017

In [5]:
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print(tabel_validasi)

# kalau ada yang statusnya BEDA, hentikan dan periksa
ada_beda = tabel_validasi[tabel_validasi["status"].str.startswith("BEDA")]
if len(ada_beda) > 0:
    print("\nPERHATIAN: ada tahun yang tidak cocok, periksa file berikut:")
    print(ada_beda)
else:
    print("\nSemua tahun lengkap cocok dengan kolom Tahunan.")

    tahun                                   file  bulan_terisi  jumlah_12_bulan  tahunan_resmi  selisih                                             status
0    2016  Jumlah Penumpang Kereta Api, 2016.csv            12         280589.0            NaN      NaN                            Kolom Tahunan tidak ada
1    2017  Jumlah Penumpang Kereta Api, 2017.csv            12         315854.0            NaN      NaN                            Kolom Tahunan tidak ada
2    2018  Jumlah Penumpang Kereta Api, 2018.csv            12         336799.0            NaN      NaN                            Kolom Tahunan tidak ada
3    2019  Jumlah Penumpang Kereta Api, 2019.csv            12         336045.0            NaN      NaN                            Kolom Tahunan tidak ada
4    2020  Jumlah Penumpang Kereta Api, 2020.csv            12         154591.0       154591.0      0.0                                                 OK
5    2021  Jumlah Penumpang Kereta Api, 2021.csv            12        

In [6]:
target, bulan_terakhir = dp.potong_bulan_belum_rilis(target_mentah)
print("Bulan aktual terakhir:", bulan_terakhir)
print("Jumlah bulan sebelum dipotong:", len(target_mentah))
print("Jumlah bulan setelah dipotong :", len(target))

bulan_kosong = dp.cari_data_hilang(target)
print("\nBulan kosong di tengah deret:", bulan_kosong)

Bulan aktual terakhir: 2026-08
Jumlah bulan sebelum dipotong: 132
Jumlah bulan setelah dipotong : 128

Bulan kosong di tengah deret: []


In [7]:
if len(bulan_kosong) > 0:
    print("Ada data hilang, ditangani dengan interpolasi linear.")
    print("Catatan: bulan yang diisi ditandai di kolom is_imputed.")
    target = dp.tangani_data_hilang(target, metode="interpolasi")
else:
    print("Tidak ada data hilang, tidak perlu penanganan.")

Tidak ada data hilang, tidak perlu penanganan.


In [8]:
print(target["penumpang_ribu"].describe())

# 5 nilai terendah dan tertinggi
print("\n5 bulan terendah:")
print(target.nsmallest(5, "penumpang_ribu")[["year_month", "penumpang_ribu"]])
print("\n5 bulan tertinggi:")
print(target.nlargest(5, "penumpang_ribu")[["year_month", "penumpang_ribu"]])

count      128.000000
mean     23258.820312
std       6984.417629
min       5077.000000
25%      21124.500000
50%      25579.500000
75%      28257.750000
max      32467.000000
Name: penumpang_ribu, dtype: float64

5 bulan terendah:
   year_month  penumpang_ribu
52    2020-05          5077.0
66    2021-07          5102.0
51    2020-04          5138.0
67    2021-08          5947.0
53    2020-06          8591.0

5 bulan tertinggi:
    year_month  penumpang_ribu
126    2026-07         32467.0
117    2025-10         31777.0
119    2025-12         31640.0
114    2025-07         31401.0
123    2026-04         30587.0


In [9]:
path_kalender = os.path.join(FOLDER_RAW, "kalender_eksogen_krl_2013_2026.csv")
kalender = dp.bersihkan_kalender(path_kalender)

print("Ukuran kalender:", kalender.shape)
print("Rentang        :", kalender["year_month"].min(), "sampai", kalender["year_month"].max())
print()
print(kalender.head())
print("\nTipe data:")
print(kalender.dtypes)

Kolom is_covid_mar20_dec22: 1 sel diperbaiki
Ukuran kalender: (132, 6)
Rentang        : 2016-01 sampai 2026-12

  year_month  total_hari  jumlah_hari_kerja  jumlah_libur_nasional_weekday  is_lebaran  is_covid
0    2016-01          31                 20                              1           0         0
1    2016-02          29                 20                              1           0         0
2    2016-03          31                 21                              2           0         0
3    2016-04          30                 21                              0           0         0
4    2016-05          31                 20                              2           0         0

Tipe data:
year_month                       object
total_hari                        int32
jumlah_hari_kerja                 int32
jumlah_libur_nasional_weekday     int32
is_lebaran                        int32
is_covid                          int32
dtype: object


In [10]:
# setiap tahun harus punya tepat 1 bulan Lebaran
lebaran_per_tahun = kalender.groupby(kalender["year_month"].str[:4])["is_lebaran"].sum()
print("Jumlah bulan Lebaran per tahun:")
print(lebaran_per_tahun)

print("\nBulan Lebaran:", kalender.loc[kalender["is_lebaran"] == 1, "year_month"].tolist())

covid = kalender.loc[kalender["is_covid"] == 1, "year_month"]
print("\nPeriode COVID:", covid.min(), "sampai", covid.max(), f"({len(covid)} bulan)")

Jumlah bulan Lebaran per tahun:
year_month
2016    1
2017    1
2018    1
2019    1
2020    1
2021    1
2022    1
2023    1
2024    1
2025    1
2026    1
Name: is_lebaran, dtype: int32

Bulan Lebaran: ['2016-07', '2017-06', '2018-06', '2019-06', '2020-05', '2021-05', '2022-05', '2023-04', '2024-04', '2025-03', '2026-03']

Periode COVID: 2020-03 sampai 2022-12 (34 bulan)


In [11]:
data_final, kalender_future = dp.gabung_target_kalender(target, kalender)

print("data_final      :", data_final.shape)
print("kalender_future :", kalender_future.shape)
print()
print(data_final.head())
print("...")
print(data_final.tail())
print("\nKalender masa depan (bulan tanpa data aktual):")
print(kalender_future)

data_final      : (128, 8)
kalender_future : (4, 7)

  year_month    tanggal  penumpang_ribu  total_hari  jumlah_hari_kerja  jumlah_libur_nasional_weekday  is_lebaran  is_covid
0    2016-01 2016-01-01         22238.0          31                 20                              1           0         0
1    2016-02 2016-02-01         21229.0          29                 20                              1           0         0
2    2016-03 2016-03-01         23206.0          31                 21                              2           0         0
3    2016-04 2016-04-01         23149.0          30                 21                              0           0         0
4    2016-05 2016-05-01         24401.0          31                 20                              2           0         0
...
    year_month    tanggal  penumpang_ribu  total_hari  jumlah_hari_kerja  jumlah_libur_nasional_weekday  is_lebaran  is_covid
123    2026-04 2026-04-01         30587.0          30                 21 

In [12]:
dp.cek_data_final(data_final)

print("\nNilai kosong per kolom:")
print(data_final.isna().sum())

print("\nTipe data:")
print(data_final.dtypes)

Semua pengecekan data_final LOLOS.

Nilai kosong per kolom:
year_month                       0
tanggal                          0
penumpang_ribu                   0
total_hari                       0
jumlah_hari_kerja                0
jumlah_libur_nasional_weekday    0
is_lebaran                       0
is_covid                         0
dtype: int64

Tipe data:
year_month                               object
tanggal                          datetime64[ns]
penumpang_ribu                          float64
total_hari                                int32
jumlah_hari_kerja                         int32
jumlah_libur_nasional_weekday             int32
is_lebaran                                int32
is_covid                                  int32
dtype: object


In [13]:
data_final.to_csv(os.path.join(FOLDER_PROCESSED, "data_final.csv"), index=False)
kalender_future.to_csv(os.path.join(FOLDER_PROCESSED, "kalender_future.csv"), index=False)
tabel_validasi.to_csv(os.path.join(FOLDER_PROCESSED, "validasi_tahunan.csv"), index=False)

print("File tersimpan di", os.path.abspath(FOLDER_PROCESSED))
for nama in sorted(os.listdir(FOLDER_PROCESSED)):
    print(" -", nama)

File tersimpan di c:\KULIAH\SEM 5\Project Sistem Cerdas\Prediksi KRL\prediksi-penumpang-krl-jabodetabek\data\processed
 - .gitkeep
 - data_final.csv
 - kalender_future.csv
 - validasi_tahunan.csv
